# L6: Multi-agent Collaboration for Financial Analysis

In this lesson, you will learn ways for making agents collaborate with each other.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [2]:
from crewai import Agent, Task, Crew

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [3]:
import os
from utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## crewAI Tools

In [4]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [5]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time "
         "to identify trends and predict market movements.",
    backstory="Specializing in financial markets, this agent "
              "uses statistical modeling and machine learning "
              "to provide crucial insights. With a knack for data, "
              "the Data Analyst Agent is the cornerstone for "
              "informing trading decisions.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [6]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based "
         "on insights from the Data Analyst Agent.",
    backstory="Equipped with a deep understanding of financial "
              "markets and quantitative analysis, this agent "
              "devises and refines trading strategies. It evaluates "
              "the performance of different approaches to determine "
              "the most profitable and risk-averse options.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [7]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies "
         "based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, "
              "and logistical details of potential trades. By evaluating "
              "these factors, it provides well-founded suggestions for "
              "when and how trades should be executed to maximize "
              "efficiency and adherence to strategy.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [8]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
         "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
              "and market dynamics, this agent scrutinizes the potential "
              "risks of proposed trades. It offers a detailed analysis of "
              "risk exposure and suggests safeguards to ensure that "
              "trading activities align with the firm’s risk tolerance.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

## Creating Tasks

In [9]:
# Task for Data Analyst Agent: Analyze Market Data
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}). "
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection}."
    ),
    agent=data_analyst_agent,
)

In [10]:
# Task for Trading Strategy Agent: Develop Trading Strategies
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference})."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    agent=trading_strategy_agent,
)


In [11]:
# Task for Trade Advisor Agent: Plan Trade Execution
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    agent=execution_agent,
)


In [12]:
# Task for Risk Advisor Agent: Assess Trading Risks
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    agent=risk_management_agent,
)

## Creating the Crew
- The `Process` class helps to delegate the workflow to the Agents (kind of like a Manager at work)
- In the example below, it will run this hierarchically.
- `manager_llm` lets you choose the "manager" LLM you want to use.

In [13]:
from crewai import Crew, Process
from langchain_openai import ChatOpenAI

# Define the crew with agents and tasks
financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],
    
    manager_llm=ChatOpenAI(model="gpt-3.5-turbo", 
                           temperature=0.7),
    process=Process.hierarchical,
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [14]:
# Example data for kicking off the process
financial_trading_inputs = {
    'stock_selection': 'AAPL',
    'initial_capital': '100000',
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Day Trading',
    'news_impact_consideration': True
}

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [15]:
### this execution will take some time to run
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

 [DEBUG]: Working Agent: Crew Manager
 [INFO]: Starting Task: Continuously monitor and analyze market data for the selected stock (AAPL). Use statistical modeling and machine learning to identify trends and predict market movements.


> Entering new CrewAgentExecutor chain...
I need to ensure that the team is continuously monitoring and analyzing market data for AAPL to identify trends and predict market movements accurately.

Action: Delegate work to co-worker
Action Input: {"coworker": "Data Analyst", "task": "Continuously monitor and analyze market data for AAPL", "context": "The data analyst needs to use statistical modeling and machine learning to identify trends and predict market movements for AAPL."}

> Entering new CrewAgentExecutor chain...
I need to gather real-time market data for AAPL to identify trends and predict market movements accurately.

Action: Read website content
Action Input: {"website_url": "https://finance.yahoo.com/quote/AAPL"} 

Apple Inc. (AAPL) Stock Price

Final Answer: Apple Inc. (AAPL) Stock Price, News, Quote & History - Yahoo Finance News Today's news US Politics World Tech Reviews and deals Audio Computing Gaming Health Home Phones Science TVs Climate change Health Science Originals The 360 Newsletters Life Health COVID-19 Fall allergies Health news Mental health Relax Sexual health Studies The Unwind Parenting Family health So mini ways Style and beauty It Figures Unapologetically Horoscopes Shopping Buying guides Food Travel Autos Gift ideas Buying guides Entertainment Celebrity TV Movies Music How to Watch Interviews Videos Finance My Portfolio News Latest Stock Market Originals The Morning Brief Economies Tariff Updates Premium News Earnings Tech Housing Crypto Markets Stocks: Most Actives Stocks: Gainers Stocks: Losers Trending Tickers Futures World Indices US Treasury Bonds Rates Currencies Crypto Top ETFs Top Mutual Funds Options: Highest Open Interest Options: Highest Implied Volatility Sectors Basic Materials Communication 

Final Answer: Apple Inc. (AAPL) Stock Price, News, Quote & History - Yahoo Finance News Today's news US Politics World Tech Reviews and deals Audio Computing Gaming Health Home Phones Science TVs Climate change Health Science Originals The 360 Newsletters Life Health COVID-19 Fall allergies Health news Mental health Relax Sexual health Studies The Unwind Parenting Family health So mini ways Style and beauty It Figures Unapologetically Horoscopes Shopping Buying guides Food Travel Autos Gift ideas Buying guides Entertainment Celebrity TV Movies Music How to Watch Interviews Videos Finance My Portfolio News Latest Stock Market Originals The Morning Brief Economies Tariff Updates Premium News Earnings Tech Housing Crypto Markets Stocks: Most Actives Stocks: Gainers Stocks: Losers Trending Tickers Futures World Indices US Treasury Bonds Rates Currencies Crypto Top ETFs Top Mutual Funds Options: Highest Open Interest Options: Highest Implied Volatility Sectors Basic Materials Communication 

Final Answer: 
After analyzing the AAPL stock data, I have found that Apple Inc. (AAPL) is currently trading at $247.04 with a 0.49% increase. The stock has shown positive returns over the past month, six months, and one year. The company's market cap is $3.711 trillion, and it operates in the consumer electronics/technology sector. Apple offers a range of products and services, including smartphones, personal computers, tablets, wearables, and accessories, as well as subscription-based services like Apple Arcade, Apple Fitness+, Apple Music, and more. The company has a strong financial position with a profit margin of 24.30% and a return on equity of 136.52%. Overall, AAPL has been performing well and could be a potential candidate for day trading strategies with a medium risk tolerance.

> Finished chain.
 

After analyzing the AAPL stock data, I have found that Apple Inc. (AAPL) is currently trading at $247.04 with a 0.49% increase. The stock has shown positive returns over the past

Thought: 
I have gathered information on AAPL trading strategies based on recent market data. Now, I need to review the content and extract insights to develop detailed execution plans for trading AAPL.

Final Answer:
Upon reviewing the search results, I have found valuable information on AAPL trading strategies and market data from various sources. Some key points include:
- AAPL stock is up 10.6% since 30 days ago.
- The next earnings date is May 1, 2025.
- AAPL historically moved higher heading into earnings more often than not.
- MoffettNathanson revised its price target for AAPL to $184.

Based on this information, I will further analyze the data, consider market trends, and develop detailed execution plans for trading AAPL to maximize profitability and minimize risk.

> Finished chain.
 

Upon reviewing the search results, I have found valuable information on AAPL trading strategies and market data from various sources. Some key points include:
- AAPL stock is up 10.6% since 30 d

Action: Read website content
Action Input: {"website_url": "https://www.investopedia.com/terms/s/swingtrading.asp"} 

Swing Trading: Definition and the Pros and Cons for Investors
Trading
Trading Strategies
Swing Trading: Definition and the Pros and Cons for Investors
Skip to content
 Investing
 Stocks
 Cryptocurrency
 Bonds
 ETFs
 Options and Derivatives
 Commodities
 Trading
 Automated Investing
 Brokers
 Fundamental Analysis
 Markets
 View All
 Simulator
 Login / Portfolio
 Trade
 Research
 My Games
 Leaderboard
 Banking
 Savings Accounts
 Certificates of Deposit (CDs)
 Money Market Accounts
 Checking Accounts
 View All
 Personal Finance
 Budgeting and Saving
 Personal Loans
 Insurance
 Mortgages
 Credit and Debt
 Student Loans
 Taxes
 Credit Cards
 Financial Literacy
 Retirement
 View All
 Economy
 Government and Policy
 Monetary Policy
 Fiscal Policy
 Economics
 View All
 News
 Markets
 Companies
 Earnings
 CD Rates
 Mortgage Rates
 Economy
 Government
 Crypto
 ETFs
 Personal Fina

Action: Read website content
Action Input: {"website_url": "https://www.investopedia.com/terms/o/optiontrading.asp"} 

Page Not Found
Skip to content
 Investing
 Stocks
 Cryptocurrency
 Bonds
 ETFs
 Options and Derivatives
 Commodities
 Trading
 Automated Investing
 Brokers
 Fundamental Analysis
 Markets
 View All
 Simulator
 Login / Portfolio
 Trade
 Research
 My Games
 Leaderboard
 Banking
 Savings Accounts
 Certificates of Deposit (CDs)
 Money Market Accounts
 Checking Accounts
 View All
 Personal Finance
 Budgeting and Saving
 Personal Loans
 Insurance
 Mortgages
 Credit and Debt
 Student Loans
 Taxes
 Credit Cards
 Financial Literacy
 Retirement
 View All
 Economy
 Government and Policy
 Monetary Policy
 Fiscal Policy
 Economics
 View All
 News
 Markets
 Companies
 Earnings
 CD Rates
 Mortgage Rates
 Economy
 Government
 Crypto
 ETFs
 Personal Finance
 View All
 Reviews
 Best Online Brokers
 Best Crypto Exchanges
 Best Savings Rates
 Best CD Rates
 Best Life Insurance
 Best Mortga

Final Answer: 
After evaluating the risks associated with each proposed trading strategy for AAPL, here is a detailed analysis with mitigation strategies:

1. Momentum Trading: Momentum trading involves taking advantage of trends in stock prices. Risks include sudden reversals in trends, leading to losses. Mitigation strategies include setting stop-loss orders to limit losses and closely monitoring stock price movements.

2. Swing Trading: Swing trading aims to capture short- to medium-term gains in stock prices. Risks include overnight and weekend market risks, as well as missing longer-term trends. Mitigation strategies include using technical analysis tools, setting stop-loss orders, and closely monitoring economic cycles.

3. Options Trading: Options trading involves trading options contracts rather than the underlying asset. Risks include high volatility, time decay, and the potential for losing the entire investment. Mitigation strategies include thorough research, using risk man

- Display the final result as Markdown.

In [16]:
from IPython.display import Markdown
Markdown(result)

Risk Analysis Report for AAPL Trading Strategies:

1. Momentum Trading:
- Risks: Sudden reversals in trends, leading to losses.
- Mitigation Strategies: Set stop-loss orders, closely monitor stock price movements.

2. Swing Trading:
- Risks: Overnight and weekend market risks, missing longer-term trends.
- Mitigation Strategies: Use technical analysis tools, set stop-loss orders, monitor economic cycles.

3. Options Trading:
- Risks: High volatility, time decay, potential for total loss.
- Mitigation Strategies: Thorough research, risk management tools, diversify options positions.

4. Sector Rotation:
- Risks: Incorrect predictions of economic cycles and sector performance.
- Mitigation Strategies: Stay informed on economic indicators, diversify sector investments, use sector-specific analysis tools.

5. Technical Analysis:
- Risks: False signals, market manipulation.
- Mitigation Strategies: Use multiple technical indicators, combine technical and fundamental analysis, stay updated on market news.

Overall, it is crucial to assess and mitigate risks associated with each trading strategy to make informed decisions and optimize trading outcomes for AAPL.

In [20]:
# -*- coding: utf-8 -*-
"""Modified Multi-AI Agent Collaboration for BTC Prediction with Real-time Forecasting"""

import warnings
warnings.filterwarnings('ignore')

import os


from crewai import Agent, Task, Crew, Process
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

from utils import get_openai_api_key, get_serper_api_key
  # Hypothetical ML model for prediction

# Set API Keys
openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

# AI Tools
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

# BTC Market Analyst Agent
btc_analyst = Agent(
    role="BTC Market Analyst",
    goal="Analyze BTC market data in real-time to predict price movements for the next hour.",
    backstory="Specializing in deep learning for financial forecasting, this agent processes real-time BTC market data and historical trends to predict price movements accurately.",
    verbose=True,
    allow_delegation=True,
    tools=[scrape_tool, search_tool]
)

# Task: Predict BTC Future Price
btc_price_prediction_task = Task(
    description="Analyze real-time BTC market data and predict price movements for the next 1 hour.",
    expected_output="Accurate BTC price predictions for the next 1 hour.",
    agent=btc_analyst,
)

# Crew Definition
btc_trading_crew = Crew(
    agents=[btc_analyst],
    tasks=[btc_price_prediction_task],
    manager_llm=ChatOpenAI(model="gpt-4-turbo", temperature=0.7),
    process=Process.hierarchical,
    verbose=True
)

# Execution Inputs
btc_trading_inputs = {
    'news_impact_consideration': True,
    'prediction_only': True  # Modify input to focus solely on prediction
}

# Run the Crew
result = btc_trading_crew.kickoff(inputs=btc_trading_inputs)

# Display the final result
from IPython.display import Markdown
Markdown(result)


2025-02-21 16:58:16,003 - 140038258432896 - __init__.py-__init__:518 - WARNING: Overriding of current TracerProvider is not allowed


 [DEBUG]: Working Agent: Crew Manager
 [INFO]: Starting Task: Analyze real-time BTC market data and predict price movements for the next 1 hour.


> Entering new CrewAgentExecutor chain...
Thought: To accurately predict BTC price movements for the next hour, I need comprehensive analysis of real-time BTC market data, including trends, volume, recent price changes, and potential market influencers. My co-worker, the BTC Market Analyst, has the expertise to handle this task.

Action: Delegate work to co-worker
Action Input: {"coworker": "BTC Market Analyst", "task": "Analyze real-time BTC market data", "context": "Examine the current BTC price, volume, and transaction data from major exchanges. Assess recent price trends and fluctuations over the last 24 hours, and identify any significant market news or events that could influence the BTC price within the next hour. Use analytical tools and models to forecast the price movement for the next 1 hour based on these observations."}

> Enter

Thought: The search results provide snippets from various sources with recent Bitcoin news and analysis. To ensure I have the latest and most relevant information that could impact Bitcoin's price in the next hour, I will read the content from one of the most directly relevant news links provided.

Action: Read website content
Action Input: {"website_url": "https://news.bitcoin.com/"} 

Bitcoin.com News
You need to enable JavaScript to run this app.

Thought: Given the current BTC price at approximately $98,218.54 and the recent bullish news suggesting a potential breakout toward $100,000, it seems that market sentiment is currently positive. The recent news highlights a general bullish trend in the market, with expectations of further increases. Coupled with the observed price movement, BTC might continue to see upward momentum in the next hour.

Final Answer: Considering the current Bitcoin price of around $98,218.54, the 24-hour trading volume, and recent market news predicting a po

Considering the current Bitcoin price of around $98,218.54, the 24-hour trading volume, and recent market news predicting a potential breakout toward $100,000, it is likely that Bitcoin will experience positive price movements in the next hour. Investors should keep an eye on market updates as sentiments can shift rapidly in the cryptocurrency market.